In [6]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. Load the Saturday Night Predictor Table
df_all = spark.read.table("gold_saturday_predictor_features").toPandas()

# 2. Fix the Data Type Trap!
df_all['Year'] = df_all['Year'].astype(int)

# 3. Re-create the Pace Rank (Since we dropped it during the inner join)
df_all['Race_Pace_Rank'] = df_all.groupby(['Year', 'Track'])['Pace_Delta_To_Leader'].rank(method='dense')

# 4. Create the Target Variable (Podium Contender)
# A true contender needs BOTH Track Position (Top 5 Grid) AND Long Run Speed (Top 5 Pace)
df_all['Is_Podium_Contender'] = np.where(
    (df_all['Simulated_Grid_Position'] <= 5) & (df_all['Race_Pace_Rank'] <= 5), 1, 0
)

# 5. TIME-SERIES SPLIT: Train the AI strictly on the Past (2024 & 2025)
df_train = df_all[df_all['Year'] < 2026]

# 6. Extract the Future (Australia 2026) for the Blind Test
df_test = df_all[(df_all['Year'] == 2026) & (df_all['Track'] == 'australia')].copy()

# 7. Define the New Saturday Features!
# (Notice we no longer use the Single Lap Proxy, we use ACTUAL Qualifying Deltas and Grid Position)
features = ['Simulated_Grid_Position', 'Qualifying_Delta_To_Pole', 'Pace_Delta_To_Leader']
X_train = df_train[features]
y_train = df_train['Is_Podium_Contender']

print("🧠 Booting up the Saturday Night Engine (Training on 2024-2025 history)...")
model = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)
print("✅ AI Trained!\n")

# 8. THE FINAL TEST: Ask the Oracle about Melbourne 2026!
if df_test.empty:
    print("⚠️ No data for Australia. Check your tables!")
else:
    X_live = df_test[features]
    
    # Get the confidence percentages
    probabilities = model.predict_proba(X_live)[:, 1] 

    # Format and rank the results
    df_test['Win_Probability'] = (probabilities * 100).round(1)
    predictions_aus = df_test.sort_values(by='Win_Probability', ascending=False)

    print("🏁 2026 AUSTRALIAN GRAND PRIX - SATURDAY NIGHT PREDICTION 🏁")
    print("=================================================================")
    display(predictions_aus[['Team', 'DriverNumber', 'Simulated_Grid_Position', 'Pace_Delta_To_Leader', 'Win_Probability']].head(10))

StatementMeta(, 2f9b676c-e582-40ef-af21-9b264429ecee, 8, Finished, Available, Finished, False)

🧠 Booting up the Saturday Night Engine (Training on 2024-2025 history)...


✅ AI Trained!

🏁 2026 AUSTRALIAN GRAND PRIX - SATURDAY NIGHT PREDICTION 🏁


SynapseWidget(Synapse.DataFrame, 80d163d8-763e-4f73-9db2-686967208ba1)

In [ ]:
# --- ADD THIS TO THE VERY BOTTOM OF YOUR ML NOTEBOOK ---

# 1. Grab the top 3 drivers and their probabilities
top_3 = predictions_aus.head(3)

# 2. Format a clean, readable text string for the email
email_text = "Here are the AI Predictions for this weekend:\n\n"

for index, row in top_3.iterrows():
    email_text += f"🏎️ P{row['Simulated_Grid_Position']}: {row['Team']} (Car {row['DriverNumber']}) - {row['Win_Probability']}% chance to win\n"

email_text += "\nGood luck!"

# 3. Use mssparkutils to push this text string back to the Pipeline
mssparkutils.notebook.exit(email_text)